In [ ]:
import os, glob, json, zipfile, random
import pandas as pd
from typing import Dict, Any, Iterable, Optional, List

# ====== EDIT HERE ======
ZIP_DIR = "../data/raw/clickbait_labels"          # 42개 zip 폴더
OUT_DIR = "../evidence/data_quality"     # Evidence 저장 폴더
os.makedirs(OUT_DIR, exist_ok=True)
# =======================

def iter_json_objects_from_bytes(b: bytes) -> Iterable[Dict[str, Any]]:
    """
    ZIP 내부 JSON bytes -> dict 스트림
    지원: 단일 dict / list[dict] / JSONL
    """
    s = b.decode("utf-8", errors="ignore").strip()
    if not s:
        return
    # 단일 JSON 또는 배열
    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            yield obj
            return
        if isinstance(obj, list):
            for it in obj:
                if isinstance(it, dict):
                    yield it
            return
    except Exception:
        pass
    # JSONL
    for line in s.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            if isinstance(obj, dict):
                yield obj
        except Exception:
            continue

def get_nested(d: Dict[str, Any], dotted: str) -> Any:
    cur = d
    for p in dotted.split("."):
        if not isinstance(cur, dict) or p not in cur:
            return None
        cur = cur[p]
    return cur

def is_nonblank_text(v: Any) -> bool:
    return isinstance(v, str) and v.strip() != ""

def stream_all_records():
    """
    42개 ZIP 전체를 순회하며 레코드(dict) 스트리밍
    """
    zip_paths = sorted(glob.glob(os.path.join(ZIP_DIR, "*.zip")))
    if not zip_paths:
        raise FileNotFoundError(f"No zip found in: {ZIP_DIR}")

    for zpath in zip_paths:
        with zipfile.ZipFile(zpath, "r") as zf:
            for name in zf.namelist():
                if not name.lower().endswith(".json"):
                    continue
                b = zf.read(name)
                for obj in iter_json_objects_from_bytes(b):
                    yield obj


In [ ]:
# --- Completeness (title/body) ---
total = 0
title_filled = 0
body_filled = 0

for obj in stream_all_records():
    src = obj.get("sourceDataInfo", obj)
    title = src.get("newsTitle", obj.get("newsTitle"))
    body  = src.get("newsContent", obj.get("newsContent"))

    total += 1
    if is_nonblank_text(title):
        title_filled += 1
    if is_nonblank_text(body):
        body_filled += 1

title_rate = (title_filled / total) * 100 if total else 0
body_rate  = (body_filled / total) * 100 if total else 0

df_out = pd.DataFrame([{
    "total_records": total,
    "title_filled": title_filled,
    "body_filled": body_filled,
    "title_completeness_pct": round(title_rate, 2),
    "body_completeness_pct": round(body_rate, 2),
    "rule": "non-null AND non-blank after strip()"
}])

out_path = os.path.join(OUT_DIR, "evidence_completeness_overall.csv")
df_out.to_csv(out_path, index=False, encoding="utf-8-sig")

print(df_out)
print("Saved:", out_path)


In [ ]:
# --- Audit sample extraction (n=400) ---
AUDIT_N = 400
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

reservoir: List[Dict[str, Any]] = []
seen = 0

def reservoir_update(item: Dict[str, Any], seen: int, k: int):
    if seen <= k:
        reservoir.append(item)
    else:
        j = random.randint(1, seen)
        if j <= k:
            reservoir[j-1] = item

for obj in stream_all_records():
    src = obj.get("sourceDataInfo", obj)
    lab = obj.get("labeledDataInfo", obj)

    row = {
        "newsID": src.get("newsID", obj.get("newsID")),
        "newsCategory": src.get("newsCategory", obj.get("newsCategory")),
        "partNum": src.get("partNum", obj.get("partNum")),
        "processType": src.get("processType", obj.get("processType")),
        "useType": src.get("useType", obj.get("useType")),
        "newsTitle": src.get("newsTitle", obj.get("newsTitle")),
        "newsContent": src.get("newsContent", obj.get("newsContent")),
        "clickbaitClass": lab.get("clickbaitClass", obj.get("clickbaitClass")),
    }

    seen += 1
    reservoir_update(row, seen, AUDIT_N)

audit_df = pd.DataFrame(reservoir)
out_path = os.path.join(OUT_DIR, "evidence_audit_sample_400.csv")
audit_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(audit_df.head(3))
print(f"Sample size: {len(audit_df)} (from streamed={seen})")
print("Saved:", out_path)


In [ ]:
# --- Representativeness (overall n) ---
total = 0
category_counts = {}

for obj in stream_all_records():
    src = obj.get("sourceDataInfo", obj)
    cat = src.get("newsCategory", obj.get("newsCategory"))
    total += 1
    if cat is not None:
        category_counts[cat] = category_counts.get(cat, 0) + 1

df_cat = pd.DataFrame([{"newsCategory": k, "count": v} for k, v in category_counts.items()])
df_cat["ratio_pct"] = (df_cat["count"] / total * 100).round(2)
df_cat = df_cat.sort_values("count", ascending=False)

summary = pd.DataFrame([{"total_records_n": total, "note": "overall n; category distribution for reference"}])

out1 = os.path.join(OUT_DIR, "evidence_representativeness_overall.csv")
summary.to_csv(out1, index=False, encoding="utf-8-sig")

out2 = os.path.join(OUT_DIR, "evidence_category_distribution.csv")
df_cat.to_csv(out2, index=False, encoding="utf-8-sig")

print(summary)
print(df_cat.head(10))
print("Saved:", out1)
print("Saved:", out2)


In [ ]:
# --- Provenance Integrity (row count + duplicate newsID) ---
total = 0
missing_id = 0
dup_id = 0
seen = set()

for obj in stream_all_records():
    src = obj.get("sourceDataInfo", obj)
    nid = src.get("newsID", obj.get("newsID"))

    total += 1
    if nid is None or (isinstance(nid, str) and nid.strip() == ""):
        missing_id += 1
        continue

    if nid in seen:
        dup_id += 1
    else:
        seen.add(nid)

df_out = pd.DataFrame([{
    "total_records": total,
    "unique_newsID": len(seen),
    "duplicate_newsID_records": dup_id,
    "missing_newsID_records": missing_id,
    "note": "counts computed from final frozen snapshot (all zip files)"
}])

out_path = os.path.join(OUT_DIR, "evidence_provenance_rowcount.csv")
df_out.to_csv(out_path, index=False, encoding="utf-8-sig")

print(df_out)
print("Saved:", out_path)


In [ ]:
import re

# 간단 PII 패턴(최소 방어용)
RE_PHONE = re.compile(r"(01[016789])[-.\s]?\d{3,4}[-.\s]?\d{4}")
RE_EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
RE_RRN   = re.compile(r"\b\d{6}[-\s]?\d{7}\b")  # 주민번호 형태(실제론 더 엄격히 가능)

total = 0
hits = 0
samples = []

for obj in stream_all_records():
    src = obj.get("sourceDataInfo", obj)
    title = src.get("newsTitle", obj.get("newsTitle")) or ""
    body  = src.get("newsContent", obj.get("newsContent")) or ""
    text = f"{title}\n{body}"

    total += 1
    found = False
    if RE_PHONE.search(text) or RE_EMAIL.search(text) or RE_RRN.search(text):
        hits += 1
        found = True

    if found and len(samples) < 50:  # 샘플은 최대 50개만
        samples.append({
            "newsID": src.get("newsID", obj.get("newsID")),
            "hit_type": "PHONE/EMAIL/RRN",
            "title_snippet": (title[:80] if isinstance(title, str) else ""),
            "body_snippet": (body[:120] if isinstance(body, str) else ""),
        })

summary = pd.DataFrame([{
    "scanned_records": total,
    "pii_hits_records": hits,
    "rule": "regex scan (phone/email/rrn) over title+body",
}])

out1 = os.path.join(OUT_DIR, "evidence_privacy_scan_summary.csv")
summary.to_csv(out1, index=False, encoding="utf-8-sig")
print(summary)
print("Saved:", out1)

if hits > 0:
    df_samples = pd.DataFrame(samples)
    out2 = os.path.join(OUT_DIR, "evidence_privacy_hits_sample.csv")
    df_samples.to_csv(out2, index=False, encoding="utf-8-sig")
    print("Saved:", out2)
